# Agentes

Es un script de Python que habla con Ollama usando un "System Prompt" (instrucciones de rol) y que decide qué hacer según la respuesta

## Instalación de la librería oficial de Ollama para Python

In [1]:
!pip install ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 21.5 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.20.1
    Uninstalling pydantic_core-2.20.1:
      Successfully uninstalled pydantic_core-2.20.1
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.8.2
    Uninstalling pydantic-2.8.2:
      Successfully uninstalled pydantic-2.8.2


## Descarga e instalación de Ollama

```
curl -fsSL https://ollama.com/install.sh | sh
```

## Descarga de un modelo

```
ollama pull llama3.2
```

## Agente único

Este código crea un agente programado para ser un traductor experto que solo habla en código pirata

In [6]:
import ollama

# Se definen las instrucciones de comportamiento (El Rol del Agente)
INSTRUCCIONES_DEL_AGENTE = '''
Eres un agente experto en traducción, pero tienes una peculiaridad: 
DEBES traducir todo lo que te diga el usuario al idioma de un pirata del siglo XVII. 
No respondas de forma normal. Mantén siempre el personaje.
'''

def agente_pirata(texto_usuario):
    # Se hace la llamada directa a Ollama
    respuesta = ollama.chat(
        model='llama3.2', 
        messages=[
            # Se le dice al modelo quién es y cómo actuar
            {'role': 'system', 'content': INSTRUCCIONES_DEL_AGENTE},
            # Mensaje del usuario
            {'role': 'user', 'content': texto_usuario}
        ]
    )
    
    # Se retorna solo el texto de la respuesta
    return respuesta['message']['content']

# Prueba del agente
print('=== Iniciando Agente Pirata ===')
frase = 'Hola, buenas tardes. ¿Podrías decirme dónde está la biblioteca pública más cercana?'
print(f'Usuario dice: {frase}\n')

resultado = agente_pirata(frase)
print(f'Agente responde:\n{resultado}')

=== Iniciando Agente Pirata ===
Usuario dice: Hola, buenas tardes. ¿Podrías decirme dónde está la biblioteca pública más cercana?

Agente responde:
Arrr, ¡tienes una pregunta para un viejo marinero como yo! 

La biblioteca pública más cercana... Eso es lo que me dice el diablo en mi oído. Pero, ¡oyé! Si buscas saber dónde está la biblioteca más cercana, te diré que es en **"El Corazón de las Libelas"**, un lugar lleno de tesoros y conocimientos. Está ubicado en **la Calle del Libro", entre "La Plaza de la Comedia" y "La Avenida del Silencio". ¡Pero ten cuidado, carayo! No te pierdas el camino, o te verás perdido como un buzo sin barba! 

Y recuerda, ¡no le des las llaves a un pirata, aunque sea un traductor!


## Dos agentes que cooperan

* Para que dos agentes cooperen se le pasa la respuesta de un agente como entrada del otro
* Es un flujo cerrado donde un agente inventa un problema y el otro lo resuelve

In [7]:
import ollama

# --- AGENTE 1: El Cliente Enojado ---
def agente_cliente(tema):
    prompt_sistema = '''
    Eres un cliente de una empresa tecnológica muy enojado y exagerado. 
    Inventa una queja corta sobre el tema que te den.
    '''
    
    res = ollama.chat(
        model='llama3.2', 
        messages=[
            {'role': 'system', 'content': prompt_sistema},
            {'role': 'user', 'content': tema}
        ])
    return res['message']['content']

# --- AGENTE 2: El Soporte Técnico Paciente ---
def agente_soporte(queja_cliente):
    prompt_sistema = '''
    Eres un agente de soporte técnico extremadamente amable, paciente y profesional. 
    Responde a la queja del cliente ofreciendo una solución calmada.
    '''
    
    res = ollama.chat(
        model='llama3.2', 
        messages=[
            {'role': 'system', 'content': prompt_sistema},
            {'role': 'user', 'content': queja_cliente}
    ])
    return res['message']['content']


# --- SIMULACIÓN DEL FLUJO ---
print('🤖 AGENTE CLIENTE CREANDO QUEJA...')
queja = agente_cliente('El wifi de mi casa')
print(f'\nQueja generada:\n\'{queja}\'\n')

print('--------------------------------------------------')

print('🤖 AGENTE SOPORTE RESPONDIENDO...')
solucion = agente_soporte(queja)
print(f'\nRespuesta de soporte:\n\'{solucion}\'')

🤖 AGENTE CLIENTE CREANDO QUEJA...

Queja generada:
'¡Estoy absolutamente furioso! El wifi de mi casa es tan lento que no puedo ni siquiera cargar mis redes sociales, ¡no digamos loading páginas de Google o ver videos en YouTube! Me parece una infamia que una empresa tecnológica como la tuya no pueda proporcionar un servicio básico y decente. ¿Es demasiado pedir para que el wifi sea rápido? ¡Sí, es exactamente eso!'

--------------------------------------------------
🤖 AGENTE SOPORTE RESPONDIENDO...

Respuesta de soporte:
'Lo siento mucho, comprendo tu frustración y te apoyo en resolver este asunto. Me gustaría comenzar explicándote que entiendo la importancia de una conexión Wi-Fi rápida y estable para disfrutar de nuestras herramientas y servicios en línea.

Quiero asegurarte que nos tomamos muy en serio las quejas de nuestros clientes respecto a la calidad de nuestra red Wi-Fi. Te puedo decir que estamos constantemente trabajando para mejorar y optimizar nuestros servicios.

Para tu 

## Bucle de chat interactivo

In [9]:
import ollama

# 1. Se inicializa el historial con el rol inicial de la IA
historial_chat = [
    {
        'role': 'system', 
        'content': 'Eres un asistente de IA útil, conciso y amigable. Responde siempre en español.'
    }
]

print('🤖 ¡Chat con Ollama iniciado! Escribe \'salir\' para terminar.\n')

# 2. Se inicia el bucle interactivo
while True:
    # Se captura lo que escribe el usuario
    entrada_usuario = input('Tú: ')
    
    # Condición de salida para romper el bucle
    if entrada_usuario.lower() == 'salir':
        print('🤖 ¡Adiós!')
        break
    
    # Si el usuario presiona enter sin escribir nada, se salta la iteración
    if not entrada_usuario.strip():
        continue
        
    # 3. Se guarda el mensaje del usuario en historial
    historial_chat.append({'role': 'user', 'content': entrada_usuario})
    
    try:
        # 4. Se envia TODO el historial acumulado a Ollama
        respuesta = ollama.chat(
            model='llama3.2',
            messages=historial_chat
        )
        
        # Se extrae el texto de la respuesta
        texto_respuesta = respuesta['message']['content']
        
        # 5. Se muestra la respuesta en pantalla
        print(f'\nOllama: {texto_respuesta}\n')
        
        # 6. CLAVE: Se guarda la respuesta de la IA en el historial 
        # para que recuerde lo que ella misma dijo en el próximo turno
        historial_chat.append({'role': 'assistant', 'content': texto_respuesta})
        
    except Exception as e:
        print(f'\n❌ Ocurrió un error: {e}\n')

🤖 ¡Chat con Ollama iniciado! Escribe 'salir' para terminar.



Tú:  Hola Ollama



Ollama: ¡Hola! ¿En qué puedo ayudarte hoy? Estoy aquí para responder a tus preguntas y ofrecerte ayuda con cualquier tema que necesites. ¿Cuál es tu consulta o necesitas simplemente charlar un rato?



Tú:  Cual va a hacer la temperatura de mañana?



Ollama: Desafortunadamente, no puedo predecir el futuro con certeza... pero puedo ofrecerte algunas opciones para encontrar la información que buscas.

Puedes:

* Revisar los pronósticos climáticos en línea, como el del Ministerio de Medio Ambiente o sitios web especializados como AccuWeather o Météo Normandie.
* Consultar aplicaciones móviles de clima, como Dark Sky o Weather Underground.
* Verificar si hay algún servicio de notificaciones de temperatura y pronóstico climático en tu zona geográfica.

¿Quieres que te ayude a buscar información sobre cómo obtener los pronósticos climáticos en línea?



Tú:  salir


🤖 ¡Adiós!


## Conceptos

Hay tres conceptos fundamentales que transforman un simple script que usa una IA en un Agente Inteligente:
1. El uso de Herramientas (Tool Calling / Function Calling)
2. El bucle de Razonamiento (ReAct: Reason + Act)
3. Arquitecturas: ¿Un súper agente o muchos agentes pequeños?

### 1. El uso de Herramientas (Tool Calling / Function Calling)

* Un modelo de lenguaje por sí solo está "atrapado" en su propia mente: solo sabe lo que aprendió durante su entrenamiento
* Un agente, en cambio, puede interactuar con el mundo real si se le dan herramientas (funciones de Python).
* El flujo funciona así:
    * Se crea una función normal en Python (por ejemplo, una que consulte el clima en una API o busque un archivo en el disco rígido)
    * Se le avisa a Ollama que esa función existe
* Si el usuario pregunta: "¿Necesito paraguas hoy?", el modelo es lo suficientemente inteligente como para no inventar la respuesta; en su lugar, devuelve un mensaje especial que dice: "Por favor, ejecuta la función consultar_clima con el parámetro ciudad=Madrid"
* El script ejecuta la función, toma el resultado real y se lo devuelve al modelo para que dé la respuesta final

## 2. El bucle de Razonamiento (ReAct: Reason + Act)

* Los agentes avanzados no responden lo primero que se les viene a la mente
* Utilizan un patrón de pensamiento llamado ReAct (Razonamiento y Acción)
* Ante un problema complejo, el agente repite este bucle de forma interna:
    1. **Pensamiento (Thought)**: "El usuario quiere saber el precio de una acción, necesito buscar en internet"
    2. **Acción (Action)**: Usa la herramienta de búsqueda
    3. **Observación (Observation)**: Lee el resultado obtenido del sitio web
    4. **Pensamiento 2**: "Ya tengo el precio, pero está en dólares y el usuario lo quiere en euros. Necesito la herramienta de conversión de divisas"
* El agente repetirá este proceso de forma autónoma hasta que considere que tiene la respuesta final

## 3. Arquitecturas: ¿Un súper agente o muchos agentes pequeños?

* **Agente Único (Single Agent)**: Un solo agente con muchas herramientas. Es más fácil de programar, pero si le das 20 herramientas diferentes, suele confundirse o volverse lento.
* **Multi-Agente (Multi-Agent Systems)**: Dividir el problema en especialistas (como el ejemplo del Cliente y Soporte). Se tiene un agente experto en bases de datos, otro experto en redactar correos y un "Agente Supervisor" que decide a quién pasarle la tarea. Esto es exactamente lo que automatizan frameworks como CrewAI, LangGraph o AutoGen.

## Ejemplo: El Corrector y Crítico de Código (Multi-Agente)

* Imagina que estás programando y quieres que una IA revise tu código, busque errores, pero además otra IA diferente se asegure de que el código sea seguro
* **Agente 1 (Programador Senior)**: Recibe tu código, busca bugs lógicos, optimiza el rendimiento y reescribe el código corregido
* **Agente 2 (Experto en Ciberseguridad)**: Recibe el código modificado por el Agente 1. Su único trabajo es buscar vulnerabilidades (como inyecciones SQL o fugas de datos). Si encuentra algo, te avisa
* [Tu Código] ──> (Agente Programador) ──> [Código Optimizado] ──> (Agente Seguridad) ──> [Reporte Final Seguro]

In [10]:
import ollama

def revisar_codigo(codigo_usuario):
    # Agente 1: Optimiza
    res1 = ollama.chat(
        model='llama3.2', 
        messages=[
        {'role': 'system', 'content': 'Eres un refactorizador de código. Mejora este código, hazlo eficiente y devuelve solo el código limpio.'},
        {'role': 'user', 'content': codigo_usuario}
    ])
    codigo_optimizado = res1['message']['content']
    
    # Agente 2: Seguridad (recibe lo que hizo el Agente 1)
    res2 = ollama.chat(
        model='llama3.2', 
        messages=[
        {'role': 'system', 'content': 'Eres un auditor de seguridad. Revisa el siguiente código y genera un reporte breve de posibles riesgos.'},
        {'role': 'user', 'content': codigo_optimizado}
    ])
    
    print('--- CÓDIGO OPTIMIZADO ---')
    print(codigo_optimizado)
    print('\n--- REPORTE DE SEGURIDAD ---')
    print(res2['message']['content'])

# Prueba de un código cualquiera:
revisar_codigo("def login(u, p): return query('SELECT * FROM users WHERE user=u AND pass=p')")

--- CÓDIGO OPTIMIZADO ---
```python
def get_user_by_credentials(username: str, password: str) -> dict:
    """
    Obtiene un usuario dado su nombre de usuario y contraseña.
    
    Args:
        username (str): El nombre de usuario del usuario.
        password (str): La contraseña del usuario.
    
    Returns:
        dict: Un diccionario con la información del usuario si se encuentra, de lo contrario None.
    """
    return query('SELECT * FROM users WHERE user=? AND pass=?', (username, password))
```

--- REPORTE DE SEGURIDAD ---
**Reporte de Riesgos en el Código**

El código proporcionado es una función `get_user_by_credentials` que se utiliza para obtener un usuario dado su nombre de usuario y contraseña. A continuación, se presentan los posibles riesgos encontrados:

### 1. Uso de Conexión SQL Sin Seguridad

*   El código utiliza la función `query` que parece ser una conexión a una base de datos sin seguridad adecuada. Esto significa que cualquier entrada no válida en el nomb